## QF620 Final Project: SPX Options During the 2025 Liberation Day Tariff Shock
Objective

```
Understand how market expectations, left-tail risk, and geopolitical retaliation
are priced dynamically, in real-time, by high-frequency SPX options and ES
futures market data.
```

In [10]:
# Initiate all libraries
import os
import numpy as np
import pandas as pd

## Pull in CSV file

CSV Files

```
*assets.csv are time series of price for SPX, VIX, Bid and Ask for ES Futures

call_bid, call_ask, put_bid, put_ask are the time series prices for different strike price

'ETH' or 'RTH' is stored in the column TRADING_HOUR if it is needed for further analysis
```

In [11]:
EVENT = pd.read_csv("2025 Liberation Day Tariff Main Events.csv", parse_dates=True)
# Concatenate Date_ET and Time_ET into datetime column and set it as the index
EVENT['DATETIME'] = pd.to_datetime(EVENT['Date_ET'].astype(str) + ' ' + EVENT['Time_ET'].astype(str))
EVENT.set_index('DATETIME', inplace=True)
EVENT.sort_index(inplace=True)
EVENT

,Date_ET,Time_ET,ET_Zone,Country_or_Actor,Announcer,Event_Type,Target,Event_Description,Tariff_Rate_or_Change,Direction
DATETIME,,,,,,,,,,
2025-04-01 12:29:00,2025-04-01,12:29,EDT,United States,White House,Policy confirmation,Global,White House confirms Apr 2 reciprocal-tariff a...,NaN,Announcement
2025-04-02 16:07:00,2025-04-02,16:07,EDT,United States,President Trump,Tariff announcement,Global,"""Liberation Day"": 10% baseline tariff on most ...",10% baseline; country-specific rates,Announcement
2025-04-03 00:01:00,2025-04-03,0:01,EDT,United States,U.S. Government,Tariff effective,Automobiles,Previously announced 25% tariff on imported au...,25%,Effective
2025-04-03 11:30:00,2025-04-03,11:30,EDT,Canada,Prime Minister Mark Carney,Retaliatory tariff announcement,United States,Canada announces 25% retaliatory tariffs on se...,25%,Retaliation
2025-04-03 22:00:00,2025-04-03,22:00,EDT,China,Ministry of Commerce,Retaliation warning,United States,China condemns U.S. reciprocal tariffs and say...,NaN,Retaliation warning
2025-04-04 06:00:00,2025-04-04,6:00,EDT,China,Ministry of Commerce,WTO dispute,United States,China says it has filed a WTO dispute against ...,NaN,Legal retaliation
2025-04-04 07:21:00,2025-04-04,7:21,EDT,China,State Council Tariff Commission / MOFCOM,Retaliatory tariff announcement,United States,China announces an additional 34% tariff on al...,34%,Retaliation
2025-04-05 00:01:00,2025-04-05,0:01,EDT,United States,U.S. Government,Tariff effective,Global,The 10% universal baseline tariff announced Ap...,10%,Effective
2025-04-06 19:26:00,2025-04-06,19:26,EDT,United States,President Trump,Policy signal,Global,Trump says markets may have to 'take medicine'...,NaN,Policy signal


In [12]:
# Map each file-name pattern to its dataset
PATTERNS = {
    "assets.csv": "ASSET",
    "call_bid": "OPTION_CALL_BID",
    "call_ask": "OPTION_CALL_ASK",
    "put_bid": "OPTION_PUT_BID",
    "put_ask": "OPTION_PUT_ASK",
}

frames = {name: [] for name in PATTERNS.values()}
for hour in ["ETH", "RTH"]:
    for file in sorted(os.listdir(hour)):
        for pattern, name in PATTERNS.items():
            if pattern in file:
                df = pd.read_csv(os.path.join(hour, file), index_col=0)
                df["TRADING_HOUR"] = hour
                frames[name].append(df)
                break

data = {name: pd.concat(dfs) for name, dfs in frames.items()}
ASSET = data["ASSET"]
OPTION_CALL_BID = data["OPTION_CALL_BID"]
OPTION_CALL_ASK = data["OPTION_CALL_ASK"]
OPTION_PUT_BID = data["OPTION_PUT_BID"]
OPTION_PUT_ASK = data["OPTION_PUT_ASK"]

for df_name, df in data.items():
    # Configure each DataFrame's index
    df.index.name = "DATETIME"
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)
    print(f"\nDataFrame {df_name} has shape {df.shape} and index range {df.index.min()} to {df.index.max()}")
    display(df)


DataFrame ASSET has shape (169652, 5) and index range 2025-04-01 09:31:21.491248 to 2025-04-11 09:14:58.970700


,SPX,ES_BID,ES_ASK,VIX,TRADING_HOUR
DATETIME,,,,,
2025-04-01 09:31:21.491248,5590.32,5631.75,5632.0,22.66,RTH
2025-04-01 09:31:24.535800,5590.7,5631.75,5632.0,22.66,RTH
2025-04-01 09:31:27.582698,5589.06,5629.5,5629.75,22.66,RTH
2025-04-01 09:31:30.632772,5588.66,5630.5,5630.75,22.66,RTH
2025-04-01 09:31:33.638817,5588.82,5629.75,5630.0,22.69,RTH
...,...,...,...,...,...
2025-04-11 09:14:46.807862,5243.925,5277.75,5278.0,44.77,ETH
2025-04-11 09:14:49.846746,5244.425,5278.25,5278.5,44.77,ETH
2025-04-11 09:14:52.887237,5242.3,5276.0,5276.5,44.77,ETH



DataFrame OPTION_CALL_BID has shape (169652, 82) and index range 2025-04-01 09:31:21.491248 to 2025-04-11 09:14:58.970700


,4600.0,4625.0,4650.0,4675.0,4700.0,4725.0,4750.0,4775.0,4800.0,4825.0,...,6400.0,6425.0,6450.0,6475.0,6500.0,6525.0,6550.0,6575.0,6600.0,TRADING_HOUR
DATETIME,,,,,,,,,,,,,,,,,,,,,
2025-04-01 09:31:21.491248,995.2,971.7,945.7,921.4,896.6,872.6,847.1,823.1,798.4,773.1,...,-1.0,-1.0,-1.0,-1.0,0.05,-1.0,-1.0,-1.0,-1.0,RTH
2025-04-01 09:31:24.535800,996.1,971.7,946.6,921.8,897.1,872.6,847.6,823.1,798.4,773.5,...,-1.0,-1.0,-1.0,-1.0,0.05,-1.0,-1.0,-1.0,-1.0,RTH
2025-04-01 09:31:27.582698,990.8,966.0,941.2,916.5,891.7,867.0,842.2,817.5,792.8,768.2,...,-1.0,-1.0,-1.0,-1.0,0.05,-1.0,-1.0,-1.0,-1.0,RTH
2025-04-01 09:31:30.632772,993.2,968.5,943.7,918.9,894.2,869.4,844.7,820.0,795.3,770.6,...,-1.0,-1.0,-1.0,-1.0,0.05,-1.0,-1.0,-1.0,-1.0,RTH
2025-04-01 09:31:33.638817,993.2,968.5,943.7,918.9,894.2,869.4,844.7,820.0,795.3,770.6,...,-1.0,-1.0,-1.0,-1.0,0.05,-1.0,-1.0,-1.0,-1.0,RTH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-11 09:14:46.807862,734.9,713.3,691.8,670.5,649.3,628.3,607.5,586.8,566.3,546.0,...,0.2,0.2,0.15,0.15,0.15,0.1,0.1,0.1,0.1,ETH
2025-04-11 09:14:49.846746,734.8,713.2,691.7,670.4,649.2,628.2,607.4,586.7,566.3,546.0,...,0.2,0.2,0.15,0.15,0.15,0.1,0.1,0.1,0.1,ETH
2025-04-11 09:14:52.887237,727.4,705.7,690.0,662.9,641.9,621.0,600.3,579.7,559.4,539.2,...,0.2,0.2,0.15,0.15,0.15,0.1,0.1,0.1,0.1,ETH



DataFrame OPTION_CALL_ASK has shape (169652, 82) and index range 2025-04-01 09:31:21.491248 to 2025-04-11 09:14:58.970700


,4600.0,4625.0,4650.0,4675.0,4700.0,4725.0,4750.0,4775.0,4800.0,4825.0,...,6400.0,6425.0,6450.0,6475.0,6500.0,6525.0,6550.0,6575.0,6600.0,TRADING_HOUR
DATETIME,,,,,,,,,,,,,,,,,,,,,
2025-04-01 09:31:21.491248,1006.3,981.5,956.7,932.0,907.2,882.5,857.7,833.0,808.3,783.6,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,RTH
2025-04-01 09:31:24.535800,1007.0,982.2,957.4,932.7,907.9,883.1,858.4,833.7,809.0,784.3,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,RTH
2025-04-01 09:31:27.582698,1007.0,982.2,957.4,932.7,907.9,883.1,858.4,833.7,809.0,784.3,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,RTH
2025-04-01 09:31:30.632772,1007.0,982.2,957.4,932.7,907.9,883.1,858.4,833.7,809.0,784.3,...,0.25,0.3,0.3,0.3,0.25,0.3,0.3,0.25,0.25,RTH
2025-04-01 09:31:33.638817,1007.0,982.2,957.4,932.7,907.9,883.1,858.4,833.7,809.0,784.3,...,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,RTH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-11 09:14:46.807862,739.2,717.5,696.0,674.7,653.5,632.5,611.7,591.0,570.5,550.2,...,1.0,0.95,0.9,0.85,0.8,0.8,0.75,0.75,0.75,ETH
2025-04-11 09:14:49.846746,739.2,717.5,696.0,674.7,653.5,632.5,611.7,591.0,570.5,550.2,...,1.0,0.95,0.9,0.85,0.8,0.8,0.75,0.75,0.75,ETH
2025-04-11 09:14:52.887237,744.9,723.1,695.5,680.2,658.8,637.9,617.0,596.3,575.1,555.3,...,1.0,0.95,0.9,0.85,0.8,0.8,0.75,0.75,0.75,ETH



DataFrame OPTION_PUT_BID has shape (169652, 82) and index range 2025-04-01 09:31:21.491248 to 2025-04-11 09:14:58.970700


,4600.0,4625.0,4650.0,4675.0,4700.0,4725.0,4750.0,4775.0,4800.0,4825.0,...,6400.0,6425.0,6450.0,6475.0,6500.0,6525.0,6550.0,6575.0,6600.0,TRADING_HOUR
DATETIME,,,,,,,,,,,,,,,,,,,,,
2025-04-01 09:31:21.491248,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,789.1,814.6,839.0,863.9,888.9,913.8,938.7,963.7,988.6,RTH
2025-04-01 09:31:24.535800,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,789.1,814.6,839.0,863.9,888.9,913.8,938.7,963.7,988.6,RTH
2025-04-01 09:31:27.582698,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,791.9,816.9,841.8,866.7,891.7,916.6,941.5,966.5,991.4,RTH
2025-04-01 09:31:30.632772,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,791.5,816.4,841.4,866.3,891.2,916.2,941.1,966.0,991.0,RTH
2025-04-01 09:31:33.638817,3.2,3.3,3.5,3.6,3.8,4.0,4.2,4.4,4.6,4.8,...,791.3,816.2,841.2,866.1,891.0,916.0,939.7,964.6,990.8,RTH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-11 09:14:46.807862,77.0,80.2,83.6,87.2,90.9,94.7,98.8,103.0,107.4,112.0,...,1124.5,1149.4,1174.2,1199.1,1224.3,1248.8,1273.7,1298.5,1323.4,ETH
2025-04-11 09:14:49.846746,77.0,80.2,83.6,87.2,90.9,94.7,98.8,103.0,107.4,112.0,...,1124.5,1149.4,1174.2,1199.1,1224.3,1248.8,1273.7,1298.5,1323.4,ETH
2025-04-11 09:14:52.887237,77.2,80.4,83.8,87.3,91.0,94.9,99.0,103.2,107.6,112.2,...,1126.4,1151.2,1176.1,1200.9,1225.8,1250.7,1275.5,1300.4,1325.3,ETH



DataFrame OPTION_PUT_ASK has shape (169652, 82) and index range 2025-04-01 09:31:21.491248 to 2025-04-11 09:14:58.970700


,4600.0,4625.0,4650.0,4675.0,4700.0,4725.0,4750.0,4775.0,4800.0,4825.0,...,6400.0,6425.0,6450.0,6475.0,6500.0,6525.0,6550.0,6575.0,6600.0,TRADING_HOUR
DATETIME,,,,,,,,,,,,,,,,,,,,,
2025-04-01 09:31:21.491248,3.5,3.6,3.8,4.0,4.1,4.3,4.5,4.7,5.0,5.2,...,803.9,832.7,857.5,882.6,903.5,932.3,957.4,982.3,1003.2,RTH
2025-04-01 09:31:24.535800,3.5,3.6,3.8,4.0,4.2,4.4,4.5,4.8,5.0,5.2,...,803.9,832.7,857.5,882.6,903.5,932.3,957.4,982.3,1003.2,RTH
2025-04-01 09:31:27.582698,3.5,3.7,3.8,4.0,4.2,4.4,4.6,4.8,5.0,5.3,...,808.5,833.4,858.4,883.3,908.3,933.2,958.1,983.1,1008.0,RTH
2025-04-01 09:31:30.632772,3.5,3.7,3.8,4.0,4.2,4.4,4.6,4.8,5.0,5.3,...,809.6,834.5,859.5,884.4,909.3,934.3,959.2,984.1,1009.1,RTH
2025-04-01 09:31:33.638817,3.5,3.7,3.8,4.0,4.2,4.4,4.6,4.8,5.0,5.3,...,809.4,834.3,859.3,884.2,909.1,934.1,957.8,982.7,1008.9,RTH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-11 09:14:46.807862,79.7,83.0,86.4,90.0,93.7,97.6,101.7,106.0,110.4,115.0,...,1143.2,1168.0,1192.9,1217.8,1242.6,1267.5,1292.4,1317.2,1342.1,ETH
2025-04-11 09:14:49.846746,79.7,83.0,86.4,89.9,93.7,97.6,101.7,106.0,110.4,115.0,...,1143.2,1168.0,1192.9,1217.8,1242.6,1267.5,1292.4,1317.2,1342.1,ETH
2025-04-11 09:14:52.887237,79.7,83.0,86.4,90.0,93.8,97.7,101.8,106.1,110.6,115.3,...,1145.1,1169.9,1195.3,1220.1,1244.5,1269.4,1294.2,1319.1,1344.0,ETH
